# Registration QC — does the brainreg → Allen registration support ENTl layer labels?

Everything in `probe_refit` judges the **probe fit** against the DiI. Nothing before this validated
the **registration** itself beyond `orientation_check.png` (axis order). This notebook runs
`registration_qc.py` (engine) and `landmark_tool.py` (check F). Rationale, conventions and the
measured numbers are written up in `REGISTRATION_QC.md`.

> **Kernel:** the **`histology`** conda env (`~/.conda/envs/histology`). Import takes ~2 min
> (the `probe_refit` → brainrender chain); keep one kernel alive for the whole run.
> Interactive sessions are a 64 GB cgroup: every check reads page ranges, never whole volumes.

**What "worked well" means here.** The freeform control grid is 400 µm, so the warp cannot see
layers: ENTl layer labels are the atlas's proportional layering carried by a smooth warp between
the two edges the registration *can* see — pia and white matter. Check E measures exactly those
at the track. Check D asks how many recorded contacts sit within registration uncertainty of the
superficial/deep boundary. Everything else is global sanity.

| check | question | function |
|---|---|---|
| A | do the atlas boundaries sit on the tissue? (overlays) | `plot_sample_overlays`, `plot_lec_zoom`, `plot_atlas_space_checker` |
| B | affine sane, truncation overhang not squash, L/R, residual yaw, tiles | `global_checks` |
| C | deformation field plausible near the track (det J) | `jacobian_check` |
| D | contacts within 50/100/150 µm of the sup/deep boundary | `contact_boundary_distances` |
| E | visible pia / grey-white edges vs atlas edges along the surface normal | `laminar_landmark_check` |
| G | along-track label coherence | `laminar_coherence` |
| F | landmark TRE (needs your clicks) | `landmark_tool` |


In [ ]:
import sys
sys.path.insert(0, '.')          # run from code/histology_refit/
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import registration_qc as rq
import landmark_tool as lt

pd.set_option('display.width', 250); pd.set_option('display.max_columns', 80)
MICE = ['ah08', 'ah09', 'ah10', 'ly05', 'ly06', 'ly07']
print('figures ->', rq.FIGURE_DIR)

## 1. Synthetic gate — run before trusting anything

Each estimator is gated in both directions on ah08: planted perturbations must be detected
**and** unperturbed truth must not be flagged. `contacts_equal_project_probe` asserts that the
slab-based labelling used here reproduces `probe_refit.project_probe` exactly.

In [ ]:
gate = rq.run_synthetic_controls('ah08')
assert gate.attrs['passed'], 'gate failed — do not trust anything below'

## 2. Phase 0 — look at it (check A) and the global table (check B)

Three figures per mouse in `data/figures/registration_qc/`:
`<mouse>_overlay_whole.png` (8 coronal levels incl. the truncation faces + sagittal/horizontal),
`<mouse>_overlay_lec.png` (the one to read: ±1.5 mm around the recorded bank, ENTl superficial vs
deep in two blues, fibre tracts yellow, DiI red, recorded bank yellow dots), and
`<mouse>_checker_atlas.png` (sample warped into atlas space vs the unfiltered atlas reference —
tests the separately estimated *inverse* warp).

In [ ]:
for m in MICE:
    for fn in (rq.plot_sample_overlays, rq.plot_lec_zoom, rq.plot_atlas_space_checker, rq.plot_lr_profile):
        plt.close(fn(m))
    rq.clear_caches(m)

In [ ]:
from IPython.display import Image, display
for m in MICE:
    display(Image(str(rq.FIGURE_DIR / f'{m}_overlay_lec.png'), width=1400))

### Record your verdict per mouse after looking at the overlays

`ok` / `suspect` / `fail`, with a note. Only you set this; the harness never overrides it.

In [ ]:
# rq.record_verdict('ah08', 'ok', 'boundaries follow the pia and the angular bundle at the track')
# rq.record_verdict('ly05', 'suspect', 'torn dorsal-left cortex; compressed band AP 4.1–5.9 mm')

In [ ]:
globals_ = {m: rq.global_checks(m) for m in MICE}
gtab = pd.DataFrame([{k: v for k, v in g.items() if not k.startswith('_')} for g in globals_.values()])
gtab[['subject', 'affine_scale_ap', 'affine_scale_dv', 'affine_scale_lr', 'affine_yaw_deg', 'affine_pitch_deg',
      'affine_roll_deg', 'residual_yaw_deg', 'residual_pitch_deg', 'missing_anterior_mm', 'mode',
      'lr_asym_max_lecHPF', 'lr_asym_worst_group', 'img_lab_asym_corr', 'ratio_ENTl', 'ratio_CA1',
      'ratio_expected', 'global_flags']]

In [ ]:
pd.concat([globals_[m]['_tables']['asymmetry'] for m in MICE]).pivot(index='group', columns='subject', values='asym').round(2)

## 3. Phase 1 — the layer question (checks D, G, E, C)

- **D** `frac_ENTl_within_100um_of_supdeep`: fraction of ENTl bank contacts whose label would move
  with a 100 µm registration error. Advisory above 0.5. `n_within_100um_of_lecmec`: contacts of the
  whole bank within 100 µm of the ENTl/ENTm (LEC/MEC) border — the border runs along the sheet and has
  no visible edge, so this count is the whole measurement; advisory above 25 % of the bank.
- **E** `laminar_*_offset_median_um`: atlas edge minus visible edge along the local surface normal
  (positive = atlas boundary deeper). The Allen annotation's own surface sits ~50 µm inside the
  template's visible edge (measured by `atlas_modality_control`), so the `_corr` columns subtract that
  baseline; a mouse reproducing +50 / 0 is registered perfectly. `laminar_thickness_ratio`
  = atlas cortical thickness / visible thickness. **`laminar_flip_frac`** = fraction of ENTl bank
  contacts whose superficial/deep label flips when their fractional depth is re-read against the
  visible edges — the direct answer.
- **C** det J near the bank: expect ≈ det A; folds (negative) or > 1 % outside [0.5, 2] flag.
- **G** flicker fraction and ordinal reversals along each shank (descriptive).

In [ ]:
locals_ = {m: rq.local_checks(m) for m in MICE}
ltab = pd.DataFrame([{k: v for k, v in l.items() if not k.startswith('_')} for l in locals_.values()])
ltab[['subject', 'n_bank_ENTl', 'frac_ENTl_within_50um_of_supdeep', 'frac_ENTl_within_100um_of_supdeep',
      'frac_ENTl_within_150um_of_supdeep',
      'n_bank_ENTm', 'n_within_100um_of_lecmec', 'n_within_150um_of_lecmec',   # LEC/MEC border fragility
      'laminar_n_with_wm',
      'laminar_pia_offset_median_um', 'laminar_wm_offset_median_um',          # raw
      'laminar_pia_offset_corr_um', 'laminar_wm_offset_corr_um',              # minus the atlas's own offsets
      'laminar_thickness_ratio_corr', 'laminar_flip_frac', 'laminar_flip_frac_corr',
      'detj_median', 'det_affine', 'detj_frac_outside', 'detj_n_negative',
      'coherence_flicker_frac', 'self_consistency_disagreement', 'local_flags', 'local_advisories']]

In [ ]:
# LEC/MEC border fragility per shank (shank 3 = posterior-most under the current convention)
pd.concat([locals_[m]['_tables']['lecmec_shanks'].assign(subject=m) for m in MICE])[['subject', 'shank', 'n', 'ENTl', 'ENTm', 'median_d_lecmec_um', 'within_100um', 'within_200um']]

In [ ]:
for m in MICE:
    lines = locals_[m]['_tables']['lines']
    if len(lines):
        plt.close(rq.plot_laminar(m, lines))
    plt.close(rq.plot_jacobian(m, locals_[m]['_tables']['jacobian']))
    rq.clear_caches(m)
for m in MICE:
    display(Image(str(rq.FIGURE_DIR / f'{m}_laminar.png'), width=1200))

In [ ]:
pd.concat([locals_[m]['_tables']['coherence'] for m in MICE])[['subject', 'shank', 'n_label_changes', 'n_flicker_runs', 'n_ordinal_reversals', 'sequence']]

## 4. Phase 2 — landmark TRE (check F, needs your clicks)

Run `%matplotlib widget` first (ipympl is installed). Click each landmark once on the **atlas**
(saved to `brainreg/atlas_landmarks.json`), then on each mouse (`<mouse>/registration_landmarks.json`).
Re-click 3–4 landmarks on one mouse to measure the click noise floor; TRE below that floor is not
interpretable.

In [ ]:
%matplotlib widget
picker = lt.LandmarkPicker('atlas').show()

In [ ]:
picker = lt.LandmarkPicker('ah08').show()

In [ ]:
%matplotlib inline
tre = pd.concat([lt.tre_table(m) for m in MICE if len(lt.tre_table(m))], ignore_index=True) if any(len(lt.tre_table(m)) for m in MICE) else pd.DataFrame()
tre_summary = pd.DataFrame([{'subject': m, **lt.tre_summary(lt.tre_table(m))} for m in MICE])
display(tre_summary)
lt.click_repeatability('ah08')
plt.close(lt.plot_tre(MICE) or plt.figure())

## 5. The table

One row per mouse, every metric, written to `data/preprocessed_data/brainreg/registration_qc.csv`.
Copy the verdict table into `REGISTRATION_QC.md`.

In [ ]:
qc = rq.qc_table(MICE)
qc